In [5]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.32.0-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached grpcio-1.74.0-cp311-cp311-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.11.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached ml_dtypes-0.5.3-cp311-cp311-m

In [4]:

import xarray as xr
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
# from IPython.display import Video, display
# import tensorflow as tf
# from tensorflow import keras


In [5]:
from pathlib import Path

data_dir = Path("/Users/amanjhurani/LocalData_SeaIce/DSO_Sea_Ice/data/CESM2_Seaiceconc")
files = sorted(data_dir.glob("*.nc"))
if not files:
    raise SystemExit(f"No files found in {data_dir}")

ds = xr.open_mfdataset([str(p) for p in files], combine="by_coords", parallel=False)
ds

/Users/amanjhurani/LocalData_SeaIce/DSO_Sea_Ice/.venv/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/Users/amanjhurani/LocalData_SeaIce/DSO_Sea_Ice/.venv/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/Users/amanjhurani/LocalData_SeaIce/DSO_Sea_Ice/.venv/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/Users/amanjhurani/LocalData_SeaIce/DSO_Sea_Ice/.venv/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: vari

<xarray.Dataset> Size: 9GB
Dimensions:    (time: 1980, nj: 384, ni: 320, d2: 2, nvertices: 4)
Coordinates:
    lat        (nj, ni) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
    lon        (nj, ni) float64 983kB dask.array<chunksize=(384, 320), meta=np.ndarray>
  * ni         (ni) int32 1kB 1 2 3 4 5 6 7 8 ... 314 315 316 317 318 319 320
  * nj         (nj) int32 2kB 1 2 3 4 5 6 7 8 ... 378 379 380 381 382 383 384
  * time       (time) object 16kB 1850-01-15 12:00:00 ... 2014-12-15 12:00:00
Dimensions without coordinates: d2, nvertices
Data variables:
    siconc     (time, nj, ni) float32 973MB dask.array<chunksize=(1, 384, 320), meta=np.ndarray>
    time_bnds  (time, d2) object 32kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    lat_bnds   (time, nj, ni, nvertices) float32 4GB dask.array<chunksize=(600, 384, 320, 4), meta=np.ndarray>
    lon_bnds   (time, nj, ni, nvertices) float32 4GB dask.array<chunksize=(600, 384, 320, 4), meta=np.ndarray>
Attributes: (12/45)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    branch_method:          standard
    branch_time_in_child:   674885.0
    branch_time_in_parent:  306600.0
    case_id:                24
    ...                     ...
    sub_experiment_id:      none
    table_id:               SImon
    tracking_id:            hdl:21.14100/15041433-3ff4-4913-8b3a-15542cefd948
    variable_id:            siconc
    variant_info:           CMIP6 20th century experiments (1850-2014) with C...
    variant_label:          r10i1p1f1

In [ ]:
SIC=ds['siconc'].values[:,320::,:]/100

N_samples=np.shape(SIC)[0]-1

X=np.zeros((np.shape(SIC)[1],np.shape(SIC)[2],1,N_samples))

Y=np.zeros((np.shape(SIC)[1],np.shape(SIC)[2],1,N_samples))

for i in range(N_samples):
    t0=np.random.randint(0, np.shape(SIC)[0]-1)
    X[:,:,0,i]=SIC[t0,:,:]
    Y[:,:,0,i]=SIC[t0+1,:,:]




In [7]:
#Save X and Y
np.savez_compressed("SIC_data.npz", X=X, Y=Y)


In [ ]:
#MSE=np.nanmean((Y_pred_persistence-Y)**2)

def create_cnn_model(input_shape, output_shape):

    x = keras.layers.Input(shape=input_shape)
    
    y = keras.layers.Conv2D(8, (3,3), activation='relu', padding='same')(x) # add a convolutional layer with a ReLU activation
    y = keras.layers.MaxPooling2D(pool_size=(2,2))(y)     # add a max pooling layer

    y = keras.layers.Conv2D(16, (3,3), activation='relu', padding='same')(y)
    y = keras.layers.MaxPooling2D(pool_size=(2,2))(y)

    y = keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(y)
    y = keras.layers.MaxPooling2D(pool_size=(2,2))(y)

    y = keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')(y)
    y = keras.layers.MaxPooling2D(pool_size=(2,2))(y)

    y = keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')(y)
    
    y = keras.layers.GlobalAveragePooling2D()(y)

    y = keras.layers.Dropout(0.5)(y)

#    y = keras.layers.Dense(128, activation='relu')(y)
    
    y = keras.layers.Dense(output_shape, activation='softmax')(y)     # add final output layer with a softmax activation
    
    model = keras.models.Model(inputs=x, outputs=y)
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model


CNN = create_cnn_model(input_shape=np.shape(X[:,:,:,-1]), output_shape=len(keep_classes))
# take a look at how the image tensor changes shape as it passes through the CNN:
CNN.summary()